# The Attention Computation

**One score matrix. One softmax. One weighted blend.**

Ep12 left us with Q, K, and V — 16 query heads, 2 key heads, 2 value heads,
plus the gate, all waiting at the input of a full-attention layer. This
notebook shows what happens when they meet: QKᵀ, the causal mask, softmax,
the weighted sum of values, the gate, and the projection back to the
residual stream.

Every step is computed manually from the real model's weights and hidden
states, then verified against the model's own output — bit for bit.

Companion to [Episode 13](https://huggingface.co/blog/EXDai/attention-mechanism).

---

## Setup

In [ ]:
import torch
import numpy as np
import os
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

os.makedirs('images', exist_ok=True)
print(f"PyTorch {torch.__version__}")

In [ ]:
import os

MODEL_NAME = "Qwen/Qwen3.6-35B-A3B"

CACHE_DIR = os.path.expanduser("~/cache/hf/hub")
if os.path.isdir(CACHE_DIR):
    os.environ["HF_HOME"] = CACHE_DIR
    os.environ["HF_HUB_CACHE"] = CACHE_DIR
    os.environ["TRANSFORMERS_CACHE"] = CACHE_DIR
    print(f"Using cache: {CACHE_DIR}")
else:
    print(f"Cache dir not found, using default")

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    # Keep the whole model in unified GPU memory. On the GB10, device_map="auto"
    # can't see GPU memory (nvidia-smi reports N/A) and spills to CPU RAM,
    # which OOMs on full forward passes.
    device_map={"": 0},
    attn_implementation="eager",  # eager so we can extract real attention weights
)
model.eval()

lm = model.model
cfg = lm.config
print(f"Hidden size:   {cfg.hidden_size}")
print(f"Num layers:    {cfg.num_hidden_layers}")
print(f"Num Q heads:   {cfg.num_attention_heads}")
print(f"Num KV heads:  {cfg.num_key_value_heads}")
print(f"Head dim:      {cfg.head_dim}")

---

## A Note on the Real Model

Qwen3.6-35B-A3B has 40 layers. Only some of them use classic full attention —
the rest use GatedDeltaNet, an efficient linear attention variant we'll cover
in a later episode. The cell below discovers which layers are which directly
from the model, the way Ep12 did.

(Ep12's article said 4 of 40. The actual config lists more — the notebook
counts them, no hardcoding.)

In this episode we focus only on the full-attention layers. Understanding the
mechanism first makes the optimization make sense.

---

## Find the First Full-Attention Layer

In [ ]:
# Full-attention layers have a self_attn module; linear layers have linear_attn
full_attn_layers = [i for i, layer in enumerate(lm.layers) if hasattr(layer, 'self_attn')]
target_layer = full_attn_layers[0]

print(f"Full-attention layers: {full_attn_layers}")
print(f"Count: {len(full_attn_layers)} of {len(lm.layers)}")
print(f"First full-attention layer: {target_layer}")
print()

attn = lm.layers[target_layer].self_attn
print(f"Module: {attn.__class__.__name__}")

---

## Run the Model Up To That Layer

Attention needs *real* hidden states — the output of all 40 layers that
came before. Raw embeddings (Ep12) won't do: each preceding layer enriches
them with context.

We run the full model forward and keep every layer's output
(`output_hidden_states=True`). The list `out.hidden_states` holds
`[embeddings, layer_0_out, layer_1_out, ...]` — so the input to layer 3 is
`hidden_states[3]`.

In [ ]:
text = "The cat sat on the"

inputs = tok(text, return_tensors="pt").to(model.device)
token_ids = inputs["input_ids"][0]
tokens = [tok.decode(tid) for tid in token_ids]

with torch.no_grad():
    out = model(
        input_ids=inputs["input_ids"],
        output_hidden_states=True,
        output_attentions=True,
    )

hs = out.hidden_states
print(f"hidden_states: {len(hs)} tensors  (embeddings + 40 layers)")
print(f"  hs[0] (embeddings):   {tuple(hs[0].shape)}")
print(f"  hs[{target_layer}] (input to layer {target_layer}): {tuple(hs[target_layer].shape)}")

x_layer = hs[target_layer][0]  # (seq, 2048) — input to the first full-attention layer
seq_len = x_layer.shape[0]
print(f"\nTokens: {tokens}")
print(f"Layer input: {x_layer.shape}  —  {seq_len} tokens × {cfg.hidden_size} dims")

# The model's own attention weights for this layer — our verification target.
# Note: output_attentions returns weights ONLY for full-attention layers, in order,
# so layer 3's weights live at index 0 in the list.
attn_pos = full_attn_layers.index(target_layer)
attn_weights_ref = out.attentions[attn_pos]
print(f"\nModel attention weights (layer {target_layer}): {tuple(attn_weights_ref.shape)}")

---

## Recap: What We're Working With

At the layer's front door, each of the 5 tokens carries a 2048-dim hidden
state. Ep12 showed how the projections turn one of those states into
Q (16 heads × 256), a gate (16 × 256), K (2 × 256) and V (2 × 256).

We now do that for *all* tokens at once — still no cross-token mixing.
The matrices are the same three from Ep12, now measured on the real layer
input instead of a raw embedding.

In [ ]:
# The layer normalizes its input before attention sees it
with torch.no_grad():
    x_normed = lm.layers[target_layer].input_layernorm(x_layer.unsqueeze(0))[0]

num_q_heads = cfg.num_attention_heads       # 16
num_kv_heads = cfg.num_key_value_heads      # 2
head_dim = cfg.head_dim                     # 256
gqa_groups = num_q_heads // num_kv_heads    # 8

with torch.no_grad():
    # Q projection doubles the width: query + gate in one shot
    qkv_out = attn.q_proj(x_normed)                            # (seq, 8192)
    q_flat, gate_flat = torch.chunk(
        qkv_out.view(seq_len, num_q_heads, head_dim * 2), 2, dim=-1
    )
    gate_flat = gate_flat.reshape(seq_len, num_q_heads * head_dim)   # (seq, 4096)

    q_pre = q_flat.transpose(0, 1)        # (16, seq, 256)  — before QK Norm
    k_pre = attn.k_proj(x_normed).view(seq_len, num_kv_heads, head_dim).transpose(0, 1)
    v_pre = attn.v_proj(x_normed).view(seq_len, num_kv_heads, head_dim).transpose(0, 1)

print(f"Q (query):   {tuple(q_flat.shape)}  —  {seq_len} tokens × {num_q_heads} heads × {head_dim}")
print(f"Gate:        {tuple(gate_flat.shape)}  —  piggybacks on the Q projection")
print(f"K:           {tuple(k_pre.shape)}  —  {num_kv_heads} KV heads × {head_dim}")
print(f"V:           {tuple(v_pre.shape)}")
print(f"GQA:         {gqa_groups} Q heads share each KV head")

---

## Step 2 — QK Norm: Equalizing Volume

Here's the problem from Ep11: the dot product `q · k` measures *both*
direction and magnitude. A key vector that happens to be long would dominate
the softmax — not because it's relevant, but because it's loud.

Ep12 showed some heads are naturally louder than others. Qwen3.6 fixes this
with RMSNorm applied to Q and K, per head, before any scores are computed.
After normalization, similarity depends on direction, not volume. V is
deliberately untouched — the model keeps control of how much content a
token carries through V's magnitude.

In [ ]:
with torch.no_grad():
    # The model's own QK Norm modules (per-head RMSNorm on the head dim)
    q_normed = attn.q_norm(q_flat).transpose(0, 1)                       # (16, seq, 256)
    k_normed = attn.k_norm(k_pre.transpose(0, 1)).transpose(0, 1)        # (2, seq, 256)
    v_heads  = v_pre                                                     # (2, seq, 256) — no norm

# Per-head L2 norms, averaged over tokens, before vs after
def head_norms(t): return t.float().norm(dim=-1).mean(dim=-1).cpu().numpy()

q_norms_before = head_norms(q_pre)
q_norms_after  = head_norms(q_normed)
k_norms_before = head_norms(k_pre)
k_norms_after  = head_norms(k_normed)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

axes[0].bar(range(num_q_heads), q_norms_before, color='steelblue', edgecolor='white')
axes[0].set_title("Q heads — BEFORE QK Norm")
axes[0].set_xlabel("Head"); axes[0].set_ylabel("Mean L2 norm")
axes[0].set_xticks(range(num_q_heads))

axes[1].bar(range(num_q_heads), q_norms_after, color='teal', edgecolor='white')
axes[1].set_title("Q heads — AFTER QK Norm")
axes[1].set_xlabel("Head"); axes[1].set_ylabel("Mean L2 norm")
axes[1].set_xticks(range(num_q_heads))

axes[2].bar([0, 1], [k_norms_before.mean(), k_norms_after.mean()],
            color=['orange', 'coral'], edgecolor='white')
axes[2].set_title("K heads — BEFORE vs AFTER")
axes[2].set_xticks([0, 1]); axes[2].set_xticklabels(["before", "after"])
axes[2].set_ylabel("Mean L2 norm")

fig.suptitle("QK Norm — Same Direction, Controlled Volume", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('images/qk_norm.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Q norms before: {q_norms_before.mean():.3f} ± {q_norms_before.std():.3f}")
print(f"Q norms after:  {q_norms_after.mean():.3f} ± {q_norms_after.std():.3f}")
print("(The residual spread comes from the learned per-dimension gamma of RMSNorm.)")

---

## Step 3 — RoPE: Position Enters the Score

Q and K are direction-normalized but *position-blind*: a token's query at
position 2 and position 4 would produce identical scores against the same key.

Ep10's mRoPE fixes this by rotation. Qwen3.6 rotates only the **first 64 of
the 256 dimensions** per head (`partial_rotary_factor = 0.25`) — the other
192 carry pure semantic content. The rotation angle depends on the token's
position, so relative position enters the dot product.

For text-only input the three mRoPE axes (temporal / height / width) all
carry the same position, so the 3D rotation collapses to classic RoPE.
We compute cos/sin the exact same way the model does.

In [ ]:
# Position ids exactly as the model builds them: (text, temporal, height, width)
position_ids = torch.arange(seq_len, device=x_layer.device).view(1, 1, -1)
position_ids = position_ids.expand(4, 1, seq_len)
rot_pids = position_ids[1:]  # the 3 mRoPE axes — identical for text

with torch.no_grad():
    cos, sin = lm.rotary_emb(x_layer.unsqueeze(0), rot_pids)

print(f"cos: {tuple(cos.shape)}  sin: {tuple(sin.shape)}  (seq × {cos.shape[-1]} rotary dims)")

# The rotation itself — 10 lines, no magic
def rotate_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(q, k, cos, sin):
    cos = cos.unsqueeze(1)  # (bs, 1, seq, 64) -> broadcast over heads
    sin = sin.unsqueeze(1)
    rotary_dim = cos.shape[-1]                       # 64
    q_rot, q_pass = q[..., :rotary_dim], q[..., rotary_dim:]
    k_rot, k_pass = k[..., :rotary_dim], k[..., rotary_dim:]
    q_rot = q_rot * cos + rotate_half(q_rot) * sin
    k_rot = k_rot * cos + rotate_half(k_rot) * sin
    return torch.cat([q_rot, q_pass], dim=-1), torch.cat([k_rot, k_pass], dim=-1)

with torch.no_grad():
    Q = q_normed.unsqueeze(0)   # (1, 16, seq, 256)
    K = k_normed.unsqueeze(0)   # (1, 2,  seq, 256)
    V = v_heads.unsqueeze(0)    # (1, 2,  seq, 256)

    Q_rot, K_rot = apply_rope(Q, K, cos, sin)
    V_rot = V  # values are not rotated — they're payload, not an address

print(f"Q after RoPE: {tuple(Q_rot.shape)}   K after RoPE: {tuple(K_rot.shape)}")

In [ ]:
# Visual proof: dims 0-63 rotate, dims 64-255 stay identical
head_show, pos_show = 0, 1  # head 0, token "cat"

q0_before = Q[0, head_show, pos_show].float().cpu().numpy()
q0_after  = Q_rot[0, head_show, pos_show].float().cpu().numpy()
delta = np.abs(q0_after - q0_before)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

axes[0].plot(q0_before[:96], alpha=0.7, label="before", lw=1.2)
axes[0].plot(q0_after[:96], alpha=0.9, label="after", lw=1.2)
axes[0].axvline(63.5, color='red', ls='--', lw=1)
axes[0].set_title(f"Q head {head_show}, token 'cat' — first 96 of 256 dims")
axes[0].legend(); axes[0].set_xlabel("dimension")

axes[1].bar(range(256), delta, width=1, color='crimson', alpha=0.7)
axes[1].axvline(63.5, color='black', ls='--', lw=1)
axes[1].set_title("|Δ| per dimension — rotation lives in dims 0-63")
axes[1].set_xlabel("dimension"); axes[1].set_ylabel("|after − before|")

freqs = torch.arange(32, device=cos.device)
cos_h = cos[0, :, :].float().cpu().numpy()  # (seq, 64) -> 32 interleaved freqs
axes[2].imshow(cos_h, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[2].set_title("cos values — 7 positions × 64 rotary dims (32 frequencies × 2)")
axes[2].set_xlabel("rotary dim"); axes[2].set_ylabel("token position")

fig.suptitle("RoPE — Rotation in the First 64 Dimensions Only", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('images/rope_rotation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Max change in dims 0-63:   {delta[:64].max():.5f}")
print(f"Max change in dims 64-255: {delta[64:].max():.2e}  (should be 0)")

---

## Step 4 — The Score Matrix (QKᵀ)

Now Q and K meet. Every query head compares its question against the keys:

1. **GQA expansion** — 8 query heads share each KV head, so K is repeated
   8× (heads 0-7 share KV head 0, heads 8-15 share KV head 1).
2. **Dot products** — `Q @ Kᵀ` gives, for every pair of tokens, how well
   the query at position i matches the key at position j.
3. **Scaling** — divide by √256 = 1/16. With 256 dimensions, the raw dot
   products have variance ~256; softmax would saturate (all scores → 0 or 1).
   Scaling keeps the scores in softmax's working range.

In [ ]:
# GQA: repeat each KV head to cover its group of query heads
K_rep = K_rot.repeat_interleave(gqa_groups, dim=1)   # (1, 16, seq, 256)
V_rep = V_rot.repeat_interleave(gqa_groups, dim=1)

scaling = head_dim ** -0.5
scores = torch.matmul(Q_rot, K_rep.transpose(2, 3)) * scaling   # (1, 16, seq, seq)

print(f"Scaling: 1/√{head_dim} = {scaling:.4f}")
print(f"Scores:  {tuple(scores.shape)}  —  {num_q_heads} heads × {seq_len}×{seq_len} token pairs")

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, h in zip(axes, [0, 3, 8, 15]):
    im = ax.imshow(scores[0, h].float().cpu().numpy(), cmap='RdBu_r', aspect='auto')
    ax.set_title(f"Head {h}  (KV head {h // gqa_groups})")
    ax.set_xticks(range(seq_len)); ax.set_xticklabels(tokens, rotation=45, fontsize=8)
    ax.set_yticks(range(seq_len)); ax.set_yticklabels(tokens, fontsize=8)
    ax.set_xlabel("key token j"); ax.set_ylabel("query token i")
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("Raw Attention Scores — QKᵀ / √256, before mask and softmax", fontsize=13, y=1.05)
plt.tight_layout()
plt.savefig('images/score_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Step 5 — The Causal Mask

The model predicts one token at a time. Token i may only look at tokens
0…i — the future must stay invisible. Anything above the diagonal is set
to −inf, which softmax will turn into exactly 0.

(During training this is enforced the same way. The mask is what makes the
upper-right triangle of every attention heatmap black.)

In [ ]:
mask = torch.full((seq_len, seq_len), float('-inf'), device=scores.device)
mask = torch.triu(mask, diagonal=1)                  # upper triangle -> -inf
mask = mask.unsqueeze(0).unsqueeze(0)                # (1, 1, seq, seq)

scored = scores + mask                               # (1, 16, seq, seq)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

axes[0].imshow(mask[0, 0].float().cpu().numpy(), cmap='gray', aspect='auto')
axes[0].set_title("The causal mask (−inf above diagonal)")
axes[0].set_xticks(range(seq_len)); axes[0].set_xticklabels(tokens, rotation=45, fontsize=8)
axes[0].set_yticks(range(seq_len)); axes[0].set_yticklabels(tokens, fontsize=8)

im = axes[1].imshow(scores[0, 0].float().cpu().numpy(), cmap='RdBu_r', aspect='auto')
axes[1].set_title("Head 0 — raw scores")
axes[1].set_xticks(range(seq_len)); axes[1].set_xticklabels(tokens, rotation=45, fontsize=8)
axes[1].set_yticks(range(seq_len)); axes[1].set_yticklabels(tokens, fontsize=8)
plt.colorbar(im, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(scored[0, 0].float().cpu().numpy(), cmap='RdBu_r', aspect='auto')
axes[2].set_title("Head 0 — after mask (future = −inf)")
axes[2].set_xticks(range(seq_len)); axes[2].set_xticklabels(tokens, rotation=45, fontsize=8)
axes[2].set_yticks(range(seq_len)); axes[2].set_yticklabels(tokens, fontsize=8)
plt.colorbar(im2, ax=axes[2], fraction=0.046)

fig.suptitle("Causal Mask — The Future Is Invisible", fontsize=13, y=1.05)
plt.tight_layout()
plt.savefig('images/causal_mask.png', dpi=150, bbox_inches='tight')
plt.show()

print("Masked entries per row:", (scored[0, 0] == float('-inf')).sum(dim=-1).tolist())

---

## Step 6 — Softmax: Scores Become Probabilities

Each row is now a competition. Softmax converts the (masked) scores into
probabilities that sum to 1 — a budget for how much of each key token's
value the query token will absorb. −inf becomes exactly 0; the strongest
score wins the largest share.

Softmax runs in float32 for stability — the whole point of the scaling in
Step 4 was to keep these numbers in a range where softmax stays sharp.

In [ ]:
probs = torch.softmax(scored, dim=-1, dtype=torch.float32)   # (1, 16, seq, seq)
probs = probs.to(Q.dtype)

print(f"Probabilities: {tuple(probs.shape)}")

# One row, before and after: token "the" (position 4, the last token), head 0
row = 4
scores_row = scores[0, 0, row].float().cpu().numpy()
probs_row  = probs[0, 0, row].float().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].bar(range(seq_len), scores_row, color='steelblue', edgecolor='white')
axes[0].set_title("Raw scores — row 'the', head 0")
axes[0].set_xticks(range(seq_len)); axes[0].set_xticklabels(tokens, rotation=45, fontsize=9)
axes[0].set_ylabel("score")

axes[1].bar(range(seq_len), probs_row, color='teal', edgecolor='white')
axes[1].set_title("After softmax — probabilities sum to 1")
axes[1].set_xticks(range(seq_len)); axes[1].set_xticklabels(tokens, rotation=45, fontsize=9)
axes[1].set_ylabel("probability")

fig.suptitle("Softmax — Raw Scores Become a Budget", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('images/softmax.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Row sums to {probs_row.sum():.4f}")
print(f"Masked positions: {probs_row[np.isneginf(scores_row)]}")

---

## Step 7 — The Weighted Sum: Context as a Blend

The probability row is the recipe. The output for token i is a **weighted
average of all value vectors** — not a selection. Take the last token,
"the" (position 4): its output is the context that will predict the next
word. It doesn't copy one token — it blends a mix of the five that came
before it.

Each head does this independently with its own recipe, producing 16
different 256-dim context vectors. The mixture is where multi-head
expressivity comes from: 16 interpretations of the same sentence.

In [ ]:
# Weighted sum: probs @ V  ->  (1, 16, seq, 256)
ctx = torch.matmul(probs, V_rep)

print(f"Context vectors: {tuple(ctx.shape)}  —  one 256-dim blend per head per token")

# The recipe for token "the" (position 4, the last token), head 0
head_show = 0
recipe = probs[0, head_show, 4].float().cpu().numpy()
blend  = ctx[0, head_show, 4].float().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].bar(range(seq_len), recipe, color='steelblue', edgecolor='white')
axes[0].set_title(f"Head {head_show} — 'the' attention recipe")
axes[0].set_xticks(range(seq_len)); axes[0].set_xticklabels(tokens, rotation=45, fontsize=9)
axes[0].set_ylabel("probability")

axes[1].plot(blend[:128], color='teal', lw=1.5)
axes[1].set_title(f"Head {head_show} — 'the' context vector (first 128 of 256 dims)")
axes[1].set_xlabel("dimension"); axes[1].set_ylabel("value")

fig.suptitle("Weighted Sum — Context Is a Mixture, Not a Choice", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('images/weighted_sum.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Recipe sums to {recipe.sum():.4f}  —  it's a convex blend of all 5 value vectors")

---

## Step 8 — The Gate: Attention's Volume Knob

Qwen3.6's signature move (Ep12): the Q projection also produces a **gate**
vector — per head, per dimension. After attention, the context is multiplied
by sigmoid(gate):

- gate → 0 ⇒ sigmoid ≈ 0 ⇒ the head is silenced, attention didn't matter
- gate → 1 ⇒ sigmoid ≈ 1 ⇒ the head's context passes through unchanged

The gate is *learned*: the model decides per head, per token, how much it
trusts that head's opinion right now.

In [ ]:
gate_act = torch.sigmoid(gate_flat)   # (seq, 4096) — per head, per dim

gated = ctx.transpose(1, 2).reshape(seq_len, num_q_heads * head_dim) * gate_act

# Mean gate per head (over tokens and dims)
gate_head_mean = gate_act.float().reshape(seq_len, num_q_heads, head_dim).mean(dim=(0, 2)).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(range(num_q_heads), gate_head_mean, color='rebeccapurple', edgecolor='white')
axes[0].set_title("Mean sigmoid(gate) per Q head")
axes[0].set_xlabel("head"); axes[0].set_ylabel("mean gate"); axes[0].set_xticks(range(num_q_heads))
axes[0].axhline(0.5, color='black', ls='--', lw=0.8)

gate_token_head = gate_act.float().reshape(seq_len, num_q_heads, head_dim).mean(dim=-1).cpu().numpy()  # (seq, 16)
im = axes[1].imshow(gate_token_head, aspect='auto', cmap='viridis')
axes[1].set_title("Gate — token × head (mean over 256 dims)")
axes[1].set_xlabel("head"); axes[1].set_ylabel("token")
axes[1].set_xticks(range(num_q_heads)); axes[1].set_yticks(range(seq_len))
axes[1].set_yticklabels(tokens, fontsize=8)
plt.colorbar(im, ax=axes[1], fraction=0.046)

fig.suptitle("The Gate — Attention's Volume Knob", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('images/gate.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Gate range: [{gate_act.min().item():.3f}, {gate_act.max().item():.3f}]")
print("Some heads are trusted almost fully; others are heavily damped.")

---

## Step 9 — The Output Projection: Back to 2048

Sixteen heads, sixteen 256-dim contexts — 4096 dimensions of opinion. The
residual stream is only 2048 wide, so W_O (2048 × 4096) squeezes the
concatenated head outputs back down. This is the "project down" from Ep11:
the model learns which information survives the bottleneck.

After this, the 2048-dim result is added to the residual stream and the
layer is done. 2048 in, 2048 out — always.

In [ ]:
with torch.no_grad():
    attn_output = attn.o_proj(gated)      # (seq, 4096) -> (seq, 2048)

print(f"Gated context: {tuple(gated.shape)}  —  concatenated heads")
print(f"After W_O:     {tuple(attn_output.shape)}  —  back to the 2048-dim residual stream")
print(f"Output norm per token: {[f'{n:.3f}' for n in attn_output.float().norm(dim=-1).tolist()]}")

---

## Verification — This IS What the Model Does

Every step above was hand-computed. Now the moment of truth: run the model's
own attention module on the same inputs and compare.

1. **Attention weights** — our manual probabilities vs the model's
   `output_attentions` (layer 3).
2. **Full output** — our gated, projected output vs the model's
   `self_attn` output.

If the pipeline is right, both differences are ~0 (only bf16 rounding).

In [ ]:
# The model's reference: feed the same normalized input through its own module
with torch.no_grad():
    ref_out, ref_weights = attn(
        x_normed.unsqueeze(0),
        position_embeddings=(cos, sin),
        attention_mask=mask,
    )

w_diff = (probs.float() - ref_weights.float()).abs()
o_diff = (attn_output.unsqueeze(0) - ref_out).abs()

print(f"Attention weights — max |manual − model|: {w_diff.max().item():.2e}")
print(f"Full output      — max |manual − model|: {o_diff.max().item():.2e}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
im = axes[0].imshow(w_diff[0, 0].cpu().numpy(), cmap='magma', aspect='auto')
axes[0].set_title("|manual probs − model probs|  (head 0)")
axes[0].set_xticks(range(seq_len)); axes[0].set_xticklabels(tokens, rotation=45, fontsize=8)
axes[0].set_yticks(range(seq_len)); axes[0].set_yticklabels(tokens, fontsize=8)
plt.colorbar(im, ax=axes[0], fraction=0.046)

im2 = axes[1].imshow(o_diff[0].float().cpu().numpy(), cmap='magma', aspect='auto')
axes[1].set_title("|manual output − model output|  (token × 2048 dims)")
axes[1].set_xlabel("dimension"); axes[1].set_ylabel("token")
axes[1].set_yticks(range(seq_len)); axes[1].set_yticklabels(tokens, fontsize=8)
plt.colorbar(im2, ax=axes[1], fraction=0.046)

fig.suptitle("Verification — Reconstructed from First Principles", fontsize=13, y=1.05)
plt.tight_layout()
plt.savefig('images/verification.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nThat is the entire attention mechanism. Nothing hidden, no magic.")

---

## What Attention Learned — Reading the Heatmaps

The mechanism is generic; the *patterns* are learned. Let's look at real
attention on a sentence with a pronoun: *"The cat sat on the floor because
it was tired."* For "it" to make sense, some head must connect it to
"cat" — and deeper layers are where semantic links form.

In [ ]:
text2 = "The cat sat on the floor because it was tired."
inputs2 = tok(text2, return_tensors="pt").to(model.device)
tokens2 = [tok.decode(t) for t in inputs2["input_ids"][0]]

with torch.no_grad():
    out2 = model(input_ids=inputs2["input_ids"], output_attentions=True)

atts2 = out2.attentions
print(f"Tokens: {tokens2}")
print(f"Attention tensors returned: {len(atts2)} (one per full-attention layer)")

# 4×4 grid: all 16 heads at the first full-attention layer (layer 3)
layer_show = full_attn_layers[0]
W = atts2[full_attn_layers.index(layer_show)][0].float().cpu().numpy()   # (16, seq, seq)

fig, axes = plt.subplots(4, 4, figsize=(15, 13))
for h, ax in enumerate(axes.flat):
    im = ax.imshow(W[h], cmap='viridis', aspect='auto', vmin=0, vmax=W[h].max())
    ax.set_title(f"Head {h}")
    ax.set_xticks(range(len(tokens2))); ax.set_xticklabels(tokens2, rotation=90, fontsize=7)
    ax.set_yticks(range(len(tokens2))); ax.set_yticklabels(tokens2, fontsize=7)
    if h % 4 != 0: ax.set_yticklabels([])
    if h < 12: ax.set_xticklabels([])
fig.suptitle(f"All 16 Heads — Layer {layer_show} — real attention on a real sentence",
             fontsize=14, y=0.99)
plt.tight_layout()
plt.savefig('images/attention_heads.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Pronoun resolution: where does "it" look, layer by layer?
it_pos  = next(i for i, t in enumerate(tokens2) if t.strip() == "it")
cat_pos = next(i for i, t in enumerate(tokens2) if t.strip() == "cat")

layers_show = full_attn_layers  # layers 3, 7, 11, ..., 39
rows = {f"L{l}": atts2[i][0, 0, it_pos].float().cpu().numpy()
        for i, l in enumerate(layers_show)}

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(tokens2))
width = 0.8 / len(layers_show)
for j, l in enumerate(layers_show):
    off = (j - len(layers_show) / 2) * width
    ax.bar(x + off, rows[f"L{l}"], width=width * 0.9, label=f"layer {l}", alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(tokens2, rotation=45, fontsize=9)
ax.set_ylabel("attention from 'it'  (head 0)")
ax.set_title("Where 'it' looks — head 0 across the full-attention layers")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.savefig('images/pronoun.png', dpi=150, bbox_inches='tight')
plt.show()

# Which layer gives "it" -> "cat" the most weight?
scores_to_cat = {l: atts2[i][0, 0, it_pos, cat_pos].item()
                 for i, l in enumerate(layers_show)}
best = max(scores_to_cat, key=scores_to_cat.get)
print(f"Attention 'it' -> 'cat' (head 0): " + ", ".join(f"L{l}={v:.3f}" for l, v in scores_to_cat.items()))
print(f"Strongest at layer {best}")

---

## The Cost of Attention

The attention core we just walked through has a weakness: **every token
pairs with every previous token**. Doubling the sequence quadruples the
QKᵀ and weighted-sum work — O(n²).

Projections are O(n) — linear in the sequence. The quadratic part is
exclusively the token-pairing steps.

In [ ]:
# FLOPs for our 5-token sentence, one full-attention layer (16 Q heads, 2 KV heads)
n = seq_len
flops_qk   = 2 * n * n * num_q_heads * head_dim
flops_av   = 2 * n * n * num_q_heads * head_dim
flops_sm   = 3 * n * n * num_q_heads          # exp + div + normalize
flops_gate = 2 * n * num_q_heads * head_dim
flops_o    = 2 * n * num_q_heads * head_dim * cfg.hidden_size

rows = [
    ("QKᵀ  (scaled dot products)",  flops_qk),
    ("softmax (exp, sum, divide)",   flops_sm),
    ("weighted sum  (probs @ V)",    flops_av),
    ("gate multiply",                flops_gate),
    ("output projection  (W_O)",     flops_o),
]
total = sum(f for _, f in rows)
print(f"Attention FLOPs for a {n}-token sequence, one layer:")
for name, f in rows:
    print(f"  {name:<40} {f:>14,}  ({f / total * 100:4.1f}%)")
print(f"  {'TOTAL':<40} {total:>14,}")
print(f"\nFor seq_len = 5, attention is tiny. For seq_len = 32K:")
print(f"  QKᵀ + weighted sum: {2 * (32_000 ** 2) * num_q_heads * head_dim * 2:,} FLOPs  — the O(n²) wall")

# O(n²) vs O(n) scaling
seqs = np.array([2 ** k for k in range(5, 15)])
full_attn = 4 * seqs ** 2 * num_q_heads * head_dim        # QKᵀ + AV  (the O(n²) core)
# projections: Q + K + V + O = 37.7M FLOPs per token (the Ep12 number), linear in n
proj_only = 2 * seqs * (num_q_heads + 2 * num_kv_heads + num_q_heads) * head_dim * cfg.hidden_size

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(seqs, full_attn, 'o-', color='crimson', label='full attention core (QKᵀ + ΣV)')
ax.loglog(seqs, proj_only, 's-', color='steelblue', label='projections (O(n))')
ax.set_xlabel("sequence length"); ax.set_ylabel("FLOPs per layer")
ax.set_title("The O(n²) Wall — Why Only Some Layers Do Full Attention")
ax.legend(); ax.grid(True, which='both', alpha=0.2)
plt.tight_layout()
plt.savefig('images/cost.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nThis is exactly why Qwen3.6 runs 10 of 40 layers as full attention")
print("(as 'synchronization points') and uses GatedDeltaNet — O(n) — for the rest.")
print("GQA also cuts KV-cache memory 8× by sharing 2 KV heads across 16 Q heads.")

---

## What We Saw

One hidden state per token, one pipeline, eight steps:

| Step | What happens | Why |
|------|--------------|-----|
| **QK Norm** | RMSNorm on Q, K per head | scores measure direction, not loudness |
| **RoPE** | rotate first 64 of 256 dims | position enters the score |
| **QKᵀ / √256** | pairwise dot products → score matrix | "how much do these tokens relate?" |
| **Causal mask** | upper triangle → −inf | the future stays invisible |
| **Softmax** | rows → probabilities summing to 1 | a budget for each token's influence |
| **Σ probs·V** | weighted blend of value vectors | context is a mixture, not a choice |
| **sigmoid(gate)** | per-head, per-dim volume knob | the model decides how much to trust |
| **W_O** | 4096 → 2048 | back to the residual stream |

Key observations:

- **The whole mechanism is linear algebra + one nonlinearity.** Dot products,
  a mask, softmax, a weighted sum, two matrix multiplies.
- **Position enters at RoPE, nowhere else.** Without rotation, "cat" at
  position 2 and "cat" at position 4 would score identically.
- **Context is a convex blend.** Every output token absorbs a weighted mix
  of all previous tokens' values — attention never copies, it mixes.
- **The gate is a learned trust knob.** Silenced heads still compute; the
  model just decides their opinion shouldn't count.
- **Attention weights are real, extractable, and match our manual math**
  to within bf16 rounding. Nothing hidden.
- **The core is O(n²).** The projections are O(n). This asymmetry explains
  why the model reserves full attention for 10 of 40 layers.

We deliberately stopped the sentence at "the". Ask the model what comes
next — it says "mat", and when it produces that token, its attention
lands on "cat" and "sat", the words it rhymes with. That's the setup for
watching attention drive generation.

**Next:** Ep14 — GatedDeltaNet — why the other 30 layers can get away with
a linear approximation, and what they trade away.